## Задание 2
*Используйте набор данных "Boston Housing" из sklearn.datasets. Используйте
Sequential Feature Selector для выбора признаков с использованием модели
Random Forest. Визуализируйте "важность" признаков.*

In [87]:
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
from sklearn.feature_selection import SequentialFeatureSelector

### Загружаем Датасет

In [88]:
df = pd.read_csv('boston_house_prices.csv')
df.describe()

,CRIM,ZN,INDUS,CHAS,NOX,RM,AGE,DIS,RAD,TAX,PTRATIO,B,LSTAT,MEDV
count,506.000000,506.000000,506.000000,506.000000,506.000000,506.000000,506.000000,506.000000,506.000000,506.000000,506.000000,506.000000,506.000000,506.000000
mean,3.613524,11.363636,11.136779,0.069170,0.554695,6.284634,68.574901,3.795043,9.549407,408.237154,18.455534,356.674032,12.653063,22.532806
std,8.601545,23.322453,6.860353,0.253994,0.115878,0.702617,28.148861,2.105710,8.707259,168.537116,2.164946,91.294864,7.141062,9.197104
min,0.006320,0.000000,0.460000,0.000000,0.385000,3.561000,2.900000,1.129600,1.000000,187.000000,12.600000,0.320000,1.730000,5.000000
25%,0.082045,0.000000,5.190000,0.000000,0.449000,5.885500,45.025000,2.100175,4.000000,279.000000,17.400000,375.377500,6.950000,17.025000
50%,0.256510,0.000000,9.690000,0.000000,0.538000,6.208500,77.500000,3.207450,5.000000,330.000000,19.050000,391.440000,11.360000,21.200000
75%,3.677083,12.500000,18.100000,0.000000,0.624000,6.623500,94.075000,5.188425,24.000000,666.000000,20.200000,396.225000,16.955000,25.000000
max,88.976200,100.000000,27.740000,1.000000,0.871000,8.780000,100.000000,12.126500,24.000000,711.000000,22.000000,396.900000,37.970000,50.000000


In [89]:
df.head()

,CRIM,ZN,INDUS,CHAS,NOX,RM,AGE,DIS,RAD,TAX,PTRATIO,B,LSTAT,MEDV
0,0.00632,18.0,2.31,0,0.538,6.575,65.2,4.0900,1,296,15.3,396.90,4.98,24.0
1,0.02731,0.0,7.07,0,0.469,6.421,78.9,4.9671,2,242,17.8,396.90,9.14,21.6
2,0.02729,0.0,7.07,0,0.469,7.185,61.1,4.9671,2,242,17.8,392.83,4.03,34.7
3,0.03237,0.0,2.18,0,0.458,6.998,45.8,6.0622,3,222,18.7,394.63,2.94,33.4
4,0.06905,0.0,2.18,0,0.458,7.147,54.2,6.0622,3,222,18.7,396.90,5.33,36.2


In [90]:
print("Все признаки:\n",df.columns.tolist())

Все признаки:
 ['CRIM', 'ZN', 'INDUS', 'CHAS', 'NOX', 'RM', 'AGE', 'DIS', 'RAD', 'TAX', 'PTRATIO', 'B', 'LSTAT', 'MEDV']


### Разделяем данные на тренировочную и тестовую выборки

In [91]:
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(
    df.drop('MEDV', axis=1),
    df['MEDV'],
    test_size=0.4,
    shuffle=True,
    random_state=42
)

### Применяем стандартизацию

In [92]:
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
x_train_scaled = sc.fit_transform(x_train)
x_test_scaled = sc.transform(x_test)

### **SFS | Sequential Feature Selector**
#### **SFS** это метод, который последовательно добавляет или удаляет признаки на основе их влияния на производительность модели. SFS существует в двух вариантах:
1. **SFS (Sequential Forward Selection)** :
    * Начинается с пустого набора признаков.
    * На каждом шаге добавляется один признак, который максимально улучшает производительность модели.
    * Процесс продолжается до тех пор, пока не будет достигнуто заданное количество признаков.
2. **SBS (Sequential Backward Selection)** :
    * Начинается со всех признаков.
    * На каждом шаге удаляется один признак, который минимально ухудшает производительность модели.
    * Процесс продолжается до тех пор, пока не останется заданное количество признаков.

https://www.geeksforgeeks.org/sequential-feature-selection/

https://scikit-learn.org/stable/modules/generated/sklearn.feature_selection.SequentialFeatureSelector.html

In [93]:
sfs = SequentialFeatureSelector(
    estimator=RandomForestRegressor(n_estimators=60, random_state=42),
    n_features_to_select=5,
    scoring='neg_mean_squared_error',
    n_jobs=-1
)
sfs.fit(x_train_scaled, y_train)
x_train_sfs = sfs.transform(x_train_scaled)
x_test_sfs = sfs.transform(x_test_scaled)

In [97]:
print(f"Отобранные признаки: {df.columns[sfs.get_support(indices=True)].tolist()}")

Отобранные признаки: ['RM', 'DIS', 'TAX', 'PTRATIO', 'LSTAT']


### Создаём модель RandomForestRegressor

https://www.geeksforgeeks.org/random-forest-regression-in-python/

In [95]:
model = RandomForestRegressor(n_estimators=20, random_state=42)
model.fit(x_train_sfs, y_train)
y_pred = model.predict(x_test_sfs)

mse_sfs = mean_squared_error(y_test, y_pred)
print("MSE модели:", mse_sfs)

MSE модели: 11.423150000000001


In [98]:
counter = -1
results = pd.DataFrame({
    'Feature': df.drop('MEDV', axis=1).columns.tolist(),
    'Selected': (sel := [1 if i else 0 for i in sfs.support_]),
    'Importance': [model.feature_importances_[counter := counter + 1] if s == 1 else 0 for s in sel]
}).sort_values('Importance', ascending=False)

print("Вывод важности признаков:")
print(results[results['Selected'] == 1].drop(["Selected"], axis=1).reset_index(drop=True))

Вывод важности признаков:
   Feature  Importance
0       RM    0.502506
1    LSTAT    0.370805
2      DIS    0.068080
3      TAX    0.030640
4  PTRATIO    0.027968


### **Итог**:
#### При помощи Sequential Feature Selector мы отобрали 5 признаков из 13, при этом оставив довольно точную модель. Блоком выше можно увидеть важность каждого из пяти отобранных признаков, в порядке убывания.